# Humedales — Gestión y Monitoreo de Humedales en Colombia

## Qué son los humedales colombianos y por qué son estratégicos

Colombia alberga más de **25 millones de hectáreas de humedales** — una de las mayores extensiones del continente — que incluyen ciénagas, lagos andinos, lagunas costeras, pantanos, turberas de páramo y los grandes sistemas inundables de la Orinoquía y la Amazonía. Son ecosistemas socioecológicos complejos: regulan inundaciones, depuran agua, almacenan carbono, sostienen pesquerías artesanales y son hogar de aves migratorias de toda América.

A pesar de su importancia, enfrentan amenazas graves: la urbanización avanza sobre humedales periurbanos (la ciénaga La Conejera en Bogotá perdió el 90 % de su espejo de agua en el siglo XX), la expansión agrícola drena los planos inundables del Caribe y la contaminación por vertimientos degrada la calidad del agua. Por eso el monitoreo continuo es una prioridad institucional: el **Sistema de Información del Recurso Hídrico (SIRH)** del SIAC centraliza datos para la toma de decisiones.

La **Convención Ramsar** (ratificada por Colombia mediante Ley 357 de 1997) obliga a reportar el estado de los sitios de importancia internacional. Colombia tiene actualmente **10 Sitios Ramsar** que cubren más de 700.000 hectáreas.

## Preguntas que este notebook responde

- ¿Cuál es la tendencia histórica del nivel de agua y los pulsos de inundación en el humedal analizado?
- ¿El régimen hidrológico está siendo alterado por la variabilidad climática ENSO (lag 3 meses)?
- ¿NSE y KGE indican que el modelo predictivo reproduce el comportamiento hidrológico observado?

## Instituciones clave

| Institución | Rol principal |
|---|---|
| MADS | Política nacional, lineamientos normativos |
| IDEAM | Monitoreo hidrológico, indicadores del SIRH |
| Instituto Humboldt (IAvH) | Investigación, inventario de humedales |
| INVEMAR | Humedales costeros y marinos |
| CARs / PNN | Gestión local, Planes de Manejo Ambiental (PMA) |

> **Contexto de dominio completo:** [`docs/fuentes/humedales.md`](../../docs/fuentes/humedales.md)  
> **Bloque:** A — Gestión | **Línea:** `humedales`  
> **Variable principal:** `nivel_agua` (m) | Rango físico: fluctuación estacional por hidroperiodo  
> **Modelos sugeridos:** SARIMA, PROPHET, XGBOOST  
> **ENSO lag:** 3 meses (`config.ENSO_LAG_MESES["humedales"]`)  
> **Métricas primarias:** NSE (Nash-Sutcliffe), KGE (Kling-Gupta Efficiency)

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "humedales"
VARIABLE = "nivel_agua"
UNIDAD = "m"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `humedales` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "humedales.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

El nivel de agua (`nivel_agua`) es la variable hidrológica fundamental del monitoreo de humedales. Se mide con **levelloggers** o reglas limnimétricas con frecuencia continua (cada 2 horas), luego se agrega a nivel diario o mensual para análisis de tendencias. El **hidroperiodo** — el patrón temporal de inundación — es el indicador más diagnóstico de la salud ecológica de un humedal.

**Variables de monitoreo recomendadas para humedales colombianos:**

| Variable | Unidad | Rango típico | Norma de referencia |
|---|---|---|---|
| Nivel del agua | m | Fluctuación estacional | Hidroperiodo natural (línea base histórica) |
| Temperatura del agua | °C | 10 - 30 °C (según altitud) | Sin norma fija — línea base local |
| Oxígeno Disuelto (OD) | mg/L | > 4 mg/L (sistemas sanos) | Res. 2115/2007 (agua potable) |
| pH | — | 6.0 - 8.5 | Res. 2115/2007 |
| Conductividad | µS/cm | < 1.000 (agua dulce) | Res. 2115/2007 |
| Cobertura macrófitas | % | 0 - 100 % | Sin norma — indicador biótico |

> **Normativa de referencia — Decreto 1076 de 2015:** compila las normas del sector ambiente incluyendo la delimitación de ronda de humedal. El POT municipal define la franja de protección. Cualquier análisis de nivel que detecte invasión de la ronda debe reportarse al MADS o la CAR regional.

> **Ruta esperada para datos reales:** `data/raw/humedales.csv`  
> Columnas mínimas: `fecha` (datetime), `nivel_agua` (m). Opcional: `od_mgl`, `ph`, `temp_agua_c`, `cobertura_macrofitas_pct`.

In [ ]:
# df = load_csv("data/raw/humedales.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "nivel_agua": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 1b. Calidad del agua y estado trofico del humedal

El **estado trofico** clasifica el nivel de enriquecimiento por nutrientes. El modelo de **Vollenweider** predice la concentracion de fosforo total en equilibrio:

```
[P] = L / (qs * (1 + tau^0.5))     L = carga de fosforo (mg/m2/ano)
qs = caudal especifico (m/ano)       tau = tiempo de residencia (anos)
```

| Estado trofico | [P] ug/L | Clorofila-a ug/L | OD (min, mg/L) |
|---|---|---|---|
| Oligotrofico | < 10 | < 2 | > 7 |
| Mesotrofico | 10 - 30 | 2 - 8 | 5 - 7 |
| Eutrofico | 30 - 100 | 8 - 25 | 3 - 5 |
| Hipertrofico | > 100 | > 25 | < 3 |

> **Norma:** Res. 196/2006 y Politica Nacional Humedales 2002. OD < 4 mg/L es umbral critico de deterioro (IDEAM).

> **Nota NDWI:** Indice de Agua de Diferencia Normalizada NDWI = (Green - NIR)/(Green + NIR) — valores > 0 indican presencia de agua libre (Sentinel-2).

In [ ]:
# Indicadores fisicoquimicos y estado trofico del humedal (simulados)
np.random.seed(42)
n = len(df)

# Oxigeno Disuelto (OD) mg/L -- desciende con eutrofizacion
od_trend = np.linspace(6.5, 4.2, n)  # deterioro gradual
od_seasonal = 0.8 * np.sin(2*np.pi*np.arange(n)/12)  # mayor OD en sequia
od = np.clip(od_trend + od_seasonal + np.random.normal(0, 0.3, n), 0.5, 12)

# Fosforo total (ug/L) -- modelo Vollenweider simplificado
L_carga = 500       # mg P/m2/ano (carga tipica humedal andino con presion agricola)
qs = 3.0            # m/ano caudal especifico
tau = 0.5           # anos tiempo de residencia
P_eq = L_carga / (qs * (1 + tau**0.5))  # Vollenweider
fosforo = np.clip(
    P_eq + np.linspace(0, 20, n) + np.random.normal(0, 5, n), 5, 200)

# Clorofila-a (ug/L) -- correlacionada con fosforo
clorofila = np.clip(0.18 * fosforo**0.9 + np.random.normal(0, 1.5, n), 0.5, 80)

# NDWI simulado (espejo de agua) -- varia con hidroperiodo
nivel_agua = df['nivel_agua'].values
ndwi = np.clip(-0.1 + 0.4*(nivel_agua - nivel_agua.min())/(nivel_agua.max()-nivel_agua.min())
               + np.random.normal(0, 0.03, n), -0.3, 0.6)

df['od_mgl'] = od.round(2)
df['fosforo_ugl'] = fosforo.round(1)
df['clorofila_ugl'] = clorofila.round(2)
df['ndwi'] = ndwi.round(3)

# Clasificacion estado trofico
def estado_trofico(p):
    if p < 10: return 'Oligotrofico'
    if p < 30: return 'Mesotrofico'
    if p < 100: return 'Eutrofico'
    return 'Hipertrofico'

df['estado_trofico'] = df['fosforo_ugl'].apply(estado_trofico)

print(f'Vollenweider [P]eq = {P_eq:.1f} ug/L ({estado_trofico(P_eq)})')
print(f'OD actual | min={od.min():.2f} max={od.max():.2f} media={od.mean():.2f} mg/L')
print(f'Meses con OD < 4 mg/L (deterioro): {(od < 4).sum()}/{n}')
print(df['estado_trofico'].value_counts())
df[['fecha','nivel_agua','od_mgl','fosforo_ugl','clorofila_ugl','ndwi','estado_trofico']].head()

## 2. Validación y EDA

La validación de datos de nivel hidrológico en humedales enfrenta retos particulares derivados del propio ecosistema:

**Por qué `validate()` con `linea_tematica="humedales"` (ADR-006):** Los rangos del nivel de agua son específicos de cada humedal y cambian con las temporadas. El validador no usa un umbral fijo sino que compara con el **hidroperiodo histórico** para detectar anomalías. Un nivel 2 m por encima del histórico en una ciénaga del Magdalena puede ser señal de La Niña o de taponamiento de canales naturales — ninguno de los dos es un error de sensor.

**Señales de alerta en el EDA de humedales:**

1. **Datos faltantes concentrados en temporada seca (dic-feb):** puede indicar que el limnógrafo quedó en seco y no registró. No imputar con la media general — usar la media histórica del mismo mes.

2. **Saltos abruptos en el nivel:** pueden indicar intervenciones humanas (apertura de compuertas, dragados, obras de encauzamiento) que deben documentarse como metadatos.

3. **Asimetría marcada en la distribución:** esperable. Los pulsos de inundación generan colas largas hacia arriba; el nivel en estiaje es relativamente estable. Una distribución simétrica puede indicar que la dinámica de inundación está siendo suprimida por obras de ingeniería.

**Sesgos geográficos a tener en cuenta:** La mayor parte del monitoreo histórico colombiano se concentra en la cuenca Magdalena-Cauca y la región Andina. Los humedales de Amazonia, Orinoquia y Pacífico están subrepresentados en el SIRH.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_humedales.html",
       title="EDA — Humedales", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

La serie de tiempo del nivel de agua de un humedal bien conservado debe mostrar un patrón **bimodal** claro, con picos en mayo (primera temporada de lluvias) y octubre-noviembre (segunda temporada), y mínimos en enero-febrero y julio-agosto. Este patrón refleja el **régimen hidrológico colombiano** controlado por la Zona de Convergencia Intertropical (ZCIT).

**Qué buscar en la visualización:**

- **Pulsos de inundación:** crestas marcadas en el hidrograma que corresponden a eventos de lluvia intensa. En humedales de llanura (Orinoquia, Caribe) estos pulsos pueden mantener el nivel elevado durante semanas.
- **Nivel de estiaje:** el mínimo anual. Si está subiendo con el tiempo puede indicar sedimentación del vaso; si está bajando, puede indicar drenaje artificial o pérdida de área.
- **Reducción progresiva de la amplitud:** si la diferencia entre nivel máximo y mínimo anual está disminuyendo, el humedal está perdiendo su dinámica hidrológica natural — señal de degradación grave.

La gráfica de medias estacionales (`plot_seasonal_means`) debe mostrar el bimodal característico. Si la región es del Pacífico o la Amazonía, el patrón puede ser diferente (unimodal o con temporada seca muy marcada).

In [ ]:
plot_series(df, "fecha", "nivel_agua", title="Humedales — nivel_agua (m)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "nivel_agua", period="month")
plt.show()

## 3c. Dashboard calidad del agua — OD, fosforo, NDWI y bioindicadores

Los **macroinvertebrados bentonicos** son bioindicadores de calidad ecologica del agua. El indice **BMWP-Col** (Biological Monitoring Working Party adaptado a Colombia) asigna puntajes a familias de macroinvertebrados segun su sensibilidad:

| BMWP-Col | Calidad | Color |
|---|---|---|
| > 150 | Muy buena | Azul |
| 101 - 150 | Buena | Verde |
| 61 - 100 | Regular | Amarillo |
| 36 - 60 | Mala | Naranja |
| < 35 | Muy mala | Rojo |

> **NDWI** (Sentinel-2): valores > 0 = agua libre; -0.1 a 0 = agua con sedimentos; < -0.1 = vegetacion/suelo seco.

In [ ]:
# Simulacion indice BMWP-Col (macroinvertebrados bentonicos)
np.random.seed(7)
# BMWP-Col desciende al deteriorarse la calidad del agua
bmwp = np.clip(
    np.linspace(95, 45, n) + np.random.normal(0, 8, n), 10, 200)
df['bmwp_col'] = bmwp.round(0).astype(int)

BMWP_COLORS = {
    'Muy buena': '#1a73e8', 'Buena': '#27ae60',
    'Regular': '#f1c40f', 'Mala': '#e67e22', 'Muy mala': '#e74c3c'}

def bmwp_categoria(b):
    if b > 150: return 'Muy buena'
    if b > 100: return 'Buena'
    if b > 60: return 'Regular'
    if b > 35: return 'Mala'
    return 'Muy mala'

df['bmwp_cat'] = df['bmwp_col'].apply(bmwp_categoria)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Panel A: OD en el tiempo
ax = axes[0, 0]
ax.plot(df['fecha'], df['od_mgl'], lw=1.5, color='#2980b9')
ax.axhline(4.0, color='red', ls='--', lw=1.5, label='Umbral critico OD=4 mg/L')
ax.axhline(7.0, color='green', ls='--', lw=1, label='Optimo OD>7 mg/L')
ax.set_title('Oxigeno Disuelto (OD)', fontweight='bold')
ax.set_ylabel('OD (mg/L)'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Panel B: Fosforo total + estado trofico
ax = axes[0, 1]
trofico_colors = df['estado_trofico'].map(
    {'Oligotrofico':'#1a73e8','Mesotrofico':'#27ae60',
     'Eutrofico':'#f1c40f','Hipertrofico':'#e74c3c'})
ax.scatter(df['fecha'], df['fosforo_ugl'], c=trofico_colors, s=15, alpha=0.7)
ax.axhline(30, color='orange', ls='--', lw=1.5, label='Limite mesotrofico 30 ug/L')
ax.set_title('Fosforo Total + Estado Trofico (Vollenweider)', fontweight='bold')
ax.set_ylabel('Fosforo (ug/L)'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Panel C: BMWP-Col (macroinvertebrados)
ax = axes[1, 0]
bmwp_c = df['bmwp_cat'].map(BMWP_COLORS)
ax.bar(df['fecha'], df['bmwp_col'], color=bmwp_c, width=20, alpha=0.85)
for thresh, label in [(35,'Muy mala'),(60,'Mala'),(100,'Regular'),(150,'Buena')]:
    ax.axhline(thresh, color='gray', ls='--', lw=0.8)
ax.set_title('BMWP-Col (macroinvertebrados bentonicos)', fontweight='bold')
ax.set_ylabel('Puntaje BMWP-Col'); ax.grid(axis='y', alpha=0.3)

# Panel D: NDWI espejo de agua
ax = axes[1, 1]
ax.plot(df['fecha'], df['ndwi'], lw=1.5, color='#3498db', label='NDWI')
ax.axhline(0.0, color='green', ls='--', lw=1.5, label='Umbral agua libre NDWI=0')
ax.fill_between(df['fecha'], df['ndwi'], 0,
                where=(df['ndwi'] > 0), alpha=0.3, color='#3498db', label='Agua libre')
ax.set_title('NDWI — Espejo de agua (Sentinel-2)', fontweight='bold')
ax.set_ylabel('NDWI'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('Dashboard Calidad Ecologica — Humedal (Ley 357/1997 Ramsar)',
             fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

print('Estado trofico actual:', df['estado_trofico'].iloc[-1])
print('BMWP-Col ultimo registro:', df['bmwp_col'].iloc[-1], '(', df['bmwp_cat'].iloc[-1], ')')
print(f'Meses con NDWI>0 (agua libre): {(df["ndwi"]>0).sum()}/{n}')

## 3b. Covariable ENSO (ONI) — lag de 3 meses en humedales

En humedales colombianos, especialmente en los sistemas de llanura de la cuenca Magdalena-Cauca y la Orinoquía, el impacto del ENSO sobre el nivel del agua es uno de los forzantes más potentes:

- **El Niño** (ONI > +0.5 °C): provoca **reducción significativa del nivel** — la ciénaga Grande de Santa Marta y el sistema del Magdalena Medio registraron en El Niño 2015-2016 niveles históricos mínimos, afectando directamente a los pescadores artesanales y a la avifauna nidificante.
- **La Niña** (ONI < -0.5 °C): provoca **inundaciones extraordinarias**. La Niña 2010-2011 generó la mayor emergencia hídrica de Colombia en décadas, afectando a 2.9 millones de personas y devastando ecosistemas de humedal por presiones hidráulicas inusuales.

**Por qué lag de 3 meses en humedales (ADR-007):** A diferencia de los páramos (lag 2 meses), los humedales de llanura responden más lentamente al forzamiento ENSO porque: (a) el caudal de los ríos que los alimentan tiene su propio tiempo de tránsito desde las cabeceras, y (b) la capacidad de amortiguación del vaso lacustre retrasa la respuesta hidrológica. La literatura colombiana (IDEAM, CORMAGDALENA) identifica un lag óptimo de **3 meses** para el nivel en ciénagas y humedales de llanura.

`enso_lagged(df, oni, linea_tematica="humedales")` agrega automáticamente `oni_lag3` y `enso_fase_lag3`. Incluir esta columna como regresor exógeno en SARIMAX mejora el RMSE entre un 15 y 30 % en series con señal ENSO clara.

In [ ]:
# --- Covariable ENSO (lag específico para Humedales) ---
oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica="humedales")
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

En la hidrología de humedales, los estadísticos descriptivos cobran significado ecológico preciso:

- **Media del nivel:** representa el nivel medio del agua en el período analizado. Comparar con registros históricos del IDEAM para detectar si el humedal está en proceso de desecación (media decreciente) o de inundación permanente (media creciente).
- **Desviación estándar:** refleja la dinámica de pulso-y-flujo del ecosistema. Una SD baja puede indicar que el humedal ha perdido su conectividad con el río alimentador — ha quedado aislado como un sistema cerrado sin dinámica estacional.
- **Asimetría (skewness):** valores positivos son esperables — la distribución del nivel tiene cola larga hacia arriba (eventos de inundación extrema). Una asimetría negativa es inusual y puede indicar problemas de drenaje artificial.
- **Percentil 10 (nivel de estiaje):** umbral operativo para alertas de desecación. Si este nivel está por debajo del mínimo histórico registrado, el ecosistema está en riesgo crítico de pérdida de espejo de agua.
- **Percentil 90 (nivel de inundación máxima):** umbral para alertas de inundación. Comparar con la cota de desbordamiento que afecta la ronda de protección definida en el POT municipal.

In [ ]:
summarize(df)

## 5. Análisis inferencial — Estacionariedad e hidroperiodo

### 5a. Prueba de estacionariedad en series de nivel (ADR-004)

Las series de nivel de agua en humedales típicamente son **no estacionarias**: tienen componente estacional fuerte (ciclo anual bimodal) y pueden tener tendencia si el ecosistema está siendo degradado o restaurado. El par ADF + KPSS es obligatorio antes de ajustar cualquier modelo ARIMA.

**Interpretación específica para humedales:**
- Si ADF rechaza H₀ (estacionaria) con p < 0.05 **y** KPSS no rechaza: la serie es estacionaria alrededor de su media. El nivel oscila pero no tiene tendencia sistemática — humedal en estado estable.
- Si KPSS rechaza H₀ (no estacionaria) con tendencia: hay un desplazamiento progresivo del nivel. Aplicar d=1 en ARIMA; investigar si hay intervención humana o cambio climático.
- La estacionalidad (período 12 meses) requiere diferenciación estacional D=1 en SARIMA.

### 5b. Mann-Kendall sobre nivel de agua

En humedales, Mann-Kendall sobre el nivel mensual detecta si existe un cambio sistemático en el hidroperiodo. Una tendencia significativa puede significar:
- **Tendencia creciente:** La Niña persistente, restauración ecológica exitosa, o reducción de extracciones de agua aguas arriba
- **Tendencia decreciente:** Degradación del humedal, aumento de extracciones, desecación por El Niño más intenso o drenaje artificial

Estos hallazgos deben reportarse en el **Plan de Manejo Ambiental (PMA)** y notificarse a la CAR responsable. La Resolución 157 de 2004 obliga a monitorear y reportar cambios en los indicadores hidrológicos de humedales bajo plan de manejo.

In [ ]:
ts = df.set_index("fecha")["nivel_agua"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas en humedales

Para el nivel de agua en humedales no existe una norma de excedencia estricta equivalente a la calidad del aire. Sin embargo, `exceedance_report()` puede configurarse con umbrales operativos derivados del PMA del humedal:

**Para calidad del agua (si se monitorean variables fisicoquímicas):**

| Variable | Norma | Umbral de alerta | Fuente |
|---|---|---|---|
| Oxígeno Disuelto | Res. 2115/2007 | < 4 mg/L (eutrofización) | `config.NORMA_AGUA_POTABLE["od"]` |
| pH | Res. 2115/2007 | < 6.0 o > 9.0 | `config.NORMA_AGUA_POTABLE["ph"]` |
| DBO₅ | Res. 631/2015 | > 30 mg/L (vertimientos) | `config.NORMA_VERTIMIENTOS["dbo5"]` |
| Temperatura del agua | Sin norma fija | > 30 °C (humedales andinos) | Línea base local |

**Para nivel hidrológico:**
El umbral relevante es la cota de desbordamiento que afecta la ronda de protección definida en el POT municipal. Si el nivel supera este umbral, se activa el protocolo de alerta temprana del PMA. Definir este umbral en `config.py` bajo la constante correspondiente al sistema hidrográfico.

**Importante (ADR-002):** Los picos de nivel durante La Niña son eventos reales de alta magnitud — no filtrarlos automáticamente. Representan el extremo ecológico que el humedal está diseñado para absorber.

In [ ]:
rep = exceedance_report(df["nivel_agua"], variable="nivel_agua")
if rep.empty:
    print("Sin normas colombianas registradas para 'nivel_agua'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento — Imputación en series hidrológicas

Para series de nivel de agua, la imputación lineal es apropiada para brechas cortas (≤ 1 mes). Para brechas más largas:

- **Método estacional:** usar el nivel promedio del mismo mes en los años disponibles. Especialmente importante para brechas durante estiaje (el nivel no es cero, hay una cota mínima del vaso).
- **Método de estación vecina:** Si existe otro limnógrafo en el mismo sistema hídrico, usar regresión simple para imputar.
- **No imputar si la brecha supera el 20 % de la serie:** En ese caso, documentar la limitación en las conclusiones y en el PMA.

**Tratamiento de datos en seco (nivel = 0 o negativo):** Algunos levelloggers registran valores negativos cuando el sensor queda en seco o hay malfuncionamiento en estiaje extremo. Estos valores deben marcarse como `NaN` (no como cero) porque el nivel mínimo real del humedal nunca es exactamente cero — hay un vaso residual.

La decisión de imputación debe registrarse en `docs/decisiones.md` indicando: método usado, período afectado y justificación. Esto es parte del proceso MRV del monitoreo del PMA.

In [ ]:
df_clean = impute(df.copy(), cols=["nivel_agua"], method="linear")
print(f"Faltantes antes: {df["nivel_agua"].isna().sum()} | "
      f"después: {df_clean["nivel_agua"].isna().sum()}")

## 7. Modelos predictivos — NSE y KGE como métricas primarias en hidrología

### Por qué NSE y KGE en lugar de solo RMSE

Para series hidrológicas de nivel de agua, las métricas estándar de series temporales (RMSE, MAE) son necesarias pero insuficientes. La hidrología exige métricas que evalúen si el modelo captura correctamente el **comportamiento dinámico del sistema**:

**Nash-Sutcliffe Efficiency (NSE):**
- Compara el modelo contra el predictor más simple posible: la media histórica
- NSE = 1: modelo perfecto
- NSE = 0: el modelo no es mejor que usar la media histórica como predicción
- NSE < 0: la media histórica predice mejor que el modelo — el modelo es inaceptable
- Para uso en gestión de humedales: NSE ≥ 0.5 se considera satisfactorio; NSE ≥ 0.7 es bueno (criterio IDEAM)

**Kling-Gupta Efficiency (KGE):**
- Descompone el error en tres componentes: correlación, sesgo de la media, y sesgo de la variabilidad
- KGE = 1: modelo perfecto
- KGE ≥ 0.5 es aceptable para aplicaciones de gestión
- Es especialmente útil porque detecta si el modelo reproduce los picos de inundación (componente de variabilidad)

Estas métricas están disponibles en `evaluation/metrics.py`. Si el modelo tiene buen RMSE pero KGE < 0.3, probablemente está prediciendo bien el nivel promedio pero falla en capturar los pulsos de inundación — exactamente lo más relevante para la gestión del PMA.

### Selección de horizonte y modelo

Para alerta temprana de inundaciones: horizonte 1-3 meses, XGBoost o SARIMAX con ONI como covariable.
Para reportes anuales del PMA: horizonte 6-12 meses, Prophet con estacionalidad flexible.

In [ ]:
ts = df_clean.set_index("fecha")["nivel_agua"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones

Al completar el análisis con datos reales, documentar aquí:

- **Estado del hidroperiodo:** ¿La serie muestra el bimodal esperado? ¿Se conserva la amplitud histórica?
- **Tendencia Mann-Kendall:** `trend`, `p-value`, `slope` (m/mes). Interpretar en términos de degradación o restauración del ecosistema.
- **Respuesta ENSO:** correlación entre `oni_lag3` y `nivel_agua`. ¿El humedal es más sensible a El Niño o a La Niña?
- **Modelo ganador:** nombre, NSE, KGE, RMSE en walk-forward validation
- **Recomendaciones para el PMA:** ¿Se detectaron anomalías que requieren acción de gestión?

### Normativa y referencias clave

| Instrumento | Aplicación en esta línea |
|---|---|
| Ley 357 de 1997 (Ramsar) | Obligaciones de reporte para Sitios Ramsar |
| Política Nacional de Humedales 2002 | Lineamientos de conservación y uso sostenible |
| Resolución 157 de 2004 | Reglamentación de planes de manejo ambiental |
| Decreto 1076 de 2015 | Norma compilatoria — ronda de humedal y régimen de usos |
| `config.NORMA_AGUA_POTABLE` | Umbrales OD, pH, conductividad (Res. 2115/2007) |
| `config.NORMA_VERTIMIENTOS` | Umbrales DBO₅, DQO, SST (Res. 631/2015) |

---

## 9. Cómo adaptar a datos reales

**Paso 1 — Obtener datos de nivel:**
- Fuente principal: **IDEAM DHIME** — buscar estaciones limnimétricas y limnigráficas cercanas al humedal de interés
- Alternativa: monitoreo propio con levellogger (Solinst, Onset HOBO), exportar a CSV desde el software del dispositivo
- Red SIRH del SIAC: http://www.siac.gov.co/

**Paso 2 — Preparar el archivo:**
```
data/raw/humedales.csv
```
Columnas mínimas requeridas:
```
fecha,nivel_agua
2015-01-31,1.23
2015-02-28,1.45
...
```
Columnas opcionales: `od_mgl`, `ph`, `temp_agua_c`, `cobertura_macrofitas_pct`, `conductividad_us`, `estacion_id`.

**Paso 3 — Ajustar la celda de carga:**
Descomentar:
```python
df = load_csv("data/raw/humedales.csv", date_col="fecha")
```

**Paso 4 — Verificar el hidroperiodo:**
Graficar el ciclo anual promedio (`plot_seasonal_means`). Si no aparece el bimodal en una región bimodal conocida, revisar si los datos están agregados incorrectamente o si hay brechas en las temporadas de lluvia.

**Paso 5 — Configurar umbrales del PMA:**
Consultar el PMA del humedal para obtener la cota máxima y mínima de alerta. Registrar en `config.py` para usar en `exceedance_report()`.

**Paso 6 — Conectar con IDEAM DHIME vía API (opcional):**
```python
from estadistica_ambiental.io.connectors import load_ideam_dhime
df = load_ideam_dhime(codigo_estacion="21017010", variable="NI", fecha_inicio="2010-01-01")
```